In [ ]:
# EMG Low Chin Tone Detection from Sleep EEG Data
# This notebook analyzes EMG signals to detect low chin tone events

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.stats import percentile
import h5py
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")




In [ ]:

#%% [markdown]
# ## 1. Data Loading and Preprocessing#%%
def load_npz_file(npz_file):
    """Load data from npz file"""
    with np.load(npz_file) as f:
        data = f["x"]
        labels = f["y"]
        sampling_rate = f["fs"]
    return data, labels, sampling_rate

def load_npz_list_files(npz_files):
    """Load and concatenate multiple npz files"""
    data = []
    labels = []
    fs = None
    for npz_f in npz_files:
        print(f"Loading {npz_f}")
        tmp_data, tmp_labels, sampling_rate = load_npz_file(npz_f)
        if fs is None:
            fs = sampling_rate
        elif fs != sampling_rate:
            raise Exception("Sampling rates differ between files")
        data.append(tmp_data)
        labels.append(tmp_labels)
    data = np.vstack(data)
    labels = np.hstack(labels)
    return data, labels, fs

#%% [markdown]
# ## 2. EMG Signal Processing Functions

#%%
def extract_emg_channel(data, channel_idx=8):
    """
    Extract EMG channel from multi-channel data
    Args:
        data: (n_epochs, n_samples, n_channels)
        channel_idx: index of EMG channel (default=8 for 9th channel)
    Returns:
        emg_data: (n_epochs, n_samples)
    """
    return data[:, :, channel_idx]

def calculate_signal_energy(signal_segment):
    """
    Calculate energy of a signal segment
    Energy = sum of squared amplitudes
    """
    return np.sum(signal_segment ** 2)

def segment_epoch_to_windows(epoch_signal, window_size_sec=3, fs=100):
    """
    Segment 30-second epoch into smaller windows
    Args:
        epoch_signal: 1D array of 30-second signal
        window_size_sec: window size in seconds (default=3)
        fs: sampling frequency
    Returns:
        segments: list of signal segments
        energies: energy values for each segment
    """
    window_samples = int(window_size_sec * fs)
    n_windows = len(epoch_signal) // window_samples
    
    segments = []
    energies = []
    
    for i in range(n_windows):
        start_idx = i * window_samples
        end_idx = start_idx + window_samples
        segment = epoch_signal[start_idx:end_idx]
        segments.append(segment)
        energies.append(calculate_signal_energy(segment))
    
    return np.array(segments), np.array(energies)

def analyze_rem_statistics(emg_data, labels, fs=100):
    """
    Analyze EMG energy statistics in REM sleep stages
    REM sleep is typically labeled as stage 4 in standard sleep scoring
    """
    # Find REM epochs (assuming REM is label 4)
    rem_indices = np.where(labels == 4)[0]
    
    if len(rem_indices) == 0:
        print("No REM epochs found. Using all epochs for statistics.")
        rem_indices = np.arange(len(labels))
    
    all_rem_energies = []
    
    for idx in rem_indices:
        _, energies = segment_epoch_to_windows(emg_data[idx], fs=fs)
        all_rem_energies.extend(energies)
    
    all_rem_energies = np.array(all_rem_energies)
    
    # Calculate statistics
    stats = {
        'mean': np.mean(all_rem_energies),
        'median': np.median(all_rem_energies),
        'std': np.std(all_rem_energies),
        'percentile_25': np.percentile(all_rem_energies, 25),
        'percentile_50': np.percentile(all_rem_energies, 50),
        'percentile_75': np.percentile(all_rem_energies, 75)
    }
    
    return all_rem_energies, stats

def determine_threshold(rem_energies, method='percentile', percentile_value=30):
    """
    Determine threshold for low chin tone detection
    Args:
        rem_energies: energy values from REM epochs
        method: 'percentile' or 'mean_std'
        percentile_value: percentile to use as threshold (default=30)
    """
    if method == 'percentile':
        threshold = np.percentile(rem_energies, percentile_value)
    elif method == 'mean_std':
        threshold = np.mean(rem_energies) - np.std(rem_energies)
    else:
        threshold = np.median(rem_energies)
    
    return threshold

#%% [markdown]
# ## 3. Load Data

#%%
# Specify data path
data_dir = "/home/bml320/sleep-cassette/data/2018_snu_sleepEDF/npz"
npz_files = []

# List all npz files
for file in os.listdir(data_dir):
    if file.endswith(".npz"):
        npz_files.append(os.path.join(data_dir, file))

# Load first few files for analysis (adjust as needed)
npz_files = sorted(npz_files)[:5]  # Load first 5 files for demonstration

# Load data
data, labels, fs = load_npz_list_files(npz_files)
print(f"Data shape: {data.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Sampling rate: {fs} Hz")

# Sleep stage mapping
stage_dict = {
    0: "W (Wake)",
    1: "N1",
    2: "N2", 
    3: "N3",
    4: "REM"
}

#%% [markdown]
# ## 4. Extract and Analyze EMG Signal

#%%
# Extract EMG channel (9th channel, index 8)
emg_data = extract_emg_channel(data, channel_idx=8)
print(f"EMG data shape: {emg_data.shape}")

# Analyze REM statistics
rem_energies, rem_stats = analyze_rem_statistics(emg_data, labels, fs)
print("\nREM EMG Energy Statistics:")
for key, value in rem_stats.items():
    print(f"{key}: {value:.2f}")

# Determine threshold
threshold = determine_threshold(rem_energies, method='percentile', percentile_value=30)
print(f"\nLow chin tone threshold: {threshold:.2f}")

#%% [markdown]
# ## 5. Statistical Analysis and Visualization

#%%
# Collect all 3-second segment energies with their sleep stage labels
all_energies = []
all_stage_labels = []

for i in range(len(emg_data)):
    _, energies = segment_epoch_to_windows(emg_data[i], fs=fs)
    all_energies.extend(energies)
    all_stage_labels.extend([labels[i]] * len(energies))

all_energies = np.array(all_energies)
all_stage_labels = np.array(all_stage_labels)

#%%
# Plot 1: Overall energy distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram of all energies
axes[0, 0].hist(all_energies, bins=50, alpha=0.7, edgecolor='black')
axes[0, 0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.2f}')
axes[0, 0].set_xlabel('Energy')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of 3-second EMG Segment Energies')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Log scale histogram
axes[0, 1].hist(np.log10(all_energies + 1), bins=50, alpha=0.7, edgecolor='black')
axes[0, 1].axvline(np.log10(threshold + 1), color='red', linestyle='--', linewidth=2, label=f'Log Threshold')
axes[0, 1].set_xlabel('Log10(Energy + 1)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Log-scale Distribution of EMG Energies')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Box plot by sleep stage
stage_energies = {stage: [] for stage in range(5)}
for energy, stage in zip(all_energies, all_stage_labels):
    stage_energies[stage].append(energy)

box_data = [stage_energies[i] for i in range(5)]
box_labels = [stage_dict[i] for i in range(5)]

bp = axes[1, 0].boxplot(box_data, labels=box_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], sns.color_palette("husl", 5)):
    patch.set_facecolor(color)
axes[1, 0].axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.2f}')
axes[1, 0].set_xlabel('Sleep Stage')
axes[1, 0].set_ylabel('Energy')
axes[1, 0].set_title('EMG Energy Distribution by Sleep Stage')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Violin plot for better distribution visualization
parts = axes[1, 1].violinplot(box_data, positions=range(5), showmeans=True, showmedians=True)
axes[1, 1].axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.2f}')
axes[1, 1].set_xticks(range(5))
axes[1, 1].set_xticklabels(box_labels)
axes[1, 1].set_xlabel('Sleep Stage')
axes[1, 1].set_ylabel('Energy')
axes[1, 1].set_title('EMG Energy Violin Plot by Sleep Stage')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#%% [markdown]
# ## 6. Low Chin Tone Detection and Visualization

#%%
def detect_low_chin_tone(emg_epoch, threshold, fs=100):
    """
    Detect low chin tone in a 30-second epoch
    Returns segments and their classifications
    """
    segments, energies = segment_epoch_to_windows(emg_epoch, fs=fs)
    low_chin_tone = energies < threshold
    return segments, energies, low_chin_tone

def plot_epoch_with_detection(emg_epoch, segments, energies, low_chin_tone, 
                             epoch_idx, stage_label, threshold, fs=100):
    """
    Plot 30-second epoch with low chin tone detection results
    """
    fig, axes = plt.subplots(3, 1, figsize=(15, 8))
    
    time_30s = np.arange(len(emg_epoch)) / fs
    
    # Plot 1: Full 30-second EMG signal
    axes[0].plot(time_30s, emg_epoch, 'b-', linewidth=0.5)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('EMG Amplitude')
    axes[0].set_title(f'Epoch {epoch_idx} - Sleep Stage: {stage_dict[stage_label]} - Full 30s EMG Signal')
    axes[0].grid(True, alpha=0.3)
    
    # Add vertical lines to show 3-second segments
    for i in range(1, 10):
        axes[0].axvline(i * 3, color='gray', linestyle=':', alpha=0.5)
    
    # Color background based on low chin tone detection
    for i in range(10):
        if low_chin_tone[i]:
            axes[0].axvspan(i * 3, (i + 1) * 3, color='red', alpha=0.2)
        else:
            axes[0].axvspan(i * 3, (i + 1) * 3, color='green', alpha=0.1)
    
    # Plot 2: Energy values for each 3-second segment
    segment_times = np.arange(10) * 3 + 1.5  # Center of each segment
    colors = ['red' if lct else 'green' for lct in low_chin_tone]
    bars = axes[1].bar(segment_times, energies, width=2.8, color=colors, 
                       alpha=0.6, edgecolor='black')
    axes[1].axhline(threshold, color='blue', linestyle='--', linewidth=2, 
                    label=f'Threshold: {threshold:.2f}')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Energy')
    axes[1].set_title('3-second Segment Energy Values')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Binary detection results
    detection_binary = low_chin_tone.astype(int)
    axes[2].step(np.arange(11) * 3, np.concatenate([detection_binary, [detection_binary[-1]]]), 
                 where='post', linewidth=2, color='purple')
    axes[2].fill_between(np.arange(11) * 3, 0, 
                        np.concatenate([detection_binary, [detection_binary[-1]]]), 
                        step='post', alpha=0.3, color='purple')
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylabel('Low Chin Tone')
    axes[2].set_yticks([0, 1])
    axes[2].set_yticklabels(['Normal', 'Low Tone'])
    axes[2].set_title(f'Low Chin Tone Detection (Total: {np.sum(low_chin_tone)}/10 segments)')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

#%%
# Select 5 epochs to visualize (preferably including different sleep stages)
# Try to get diverse sleep stages
selected_epochs = []
stages_to_find = [0, 1, 2, 3, 4]  # All sleep stages
stages_found = {stage: False for stage in stages_to_find}

for i, label in enumerate(labels):
    if not stages_found[label]:
        selected_epochs.append(i)
        stages_found[label] = True
    if len(selected_epochs) >= 5:
        break

# If we don't have 5 different stages, just take the first 5 epochs
if len(selected_epochs) < 5:
    selected_epochs = list(range(5))

print(f"Selected epochs: {selected_epochs}")
print(f"Their stages: {[stage_dict[labels[i]] for i in selected_epochs]}")

#%%
# Plot detection results for selected epochs
for i, epoch_idx in enumerate(selected_epochs):
    emg_epoch = emg_data[epoch_idx]
    segments, energies, low_chin_tone = detect_low_chin_tone(emg_epoch, threshold, fs)
    
    fig = plot_epoch_with_detection(emg_epoch, segments, energies, low_chin_tone,
                                   epoch_idx, labels[epoch_idx], threshold, fs)
    plt.show()
    
    # Print summary for this epoch
    print(f"\nEpoch {epoch_idx} Summary:")
    print(f"  Sleep Stage: {stage_dict[labels[epoch_idx]]}")
    print(f"  Low chin tone segments: {np.sum(low_chin_tone)}/10")
    print(f"  Mean energy: {np.mean(energies):.2f}")
    print(f"  Energy range: [{np.min(energies):.2f}, {np.max(energies):.2f}]")

#%% [markdown]
# ## 7. Summary Statistics

#%%
# Calculate overall detection statistics
all_detections = []
for i in range(len(emg_data)):
    _, energies, low_chin_tone = detect_low_chin_tone(emg_data[i], threshold, fs)
    all_detections.append(np.sum(low_chin_tone))

all_detections = np.array(all_detections)

# Statistics by sleep stage
stage_detection_stats = {}
for stage in range(5):
    stage_indices = np.where(labels == stage)[0]
    if len(stage_indices) > 0:
        stage_detections = all_detections[stage_indices]
        stage_detection_stats[stage] = {
            'mean_segments': np.mean(stage_detections),
            'std_segments': np.std(stage_detections),
            'max_segments': np.max(stage_detections),
            'min_segments': np.min(stage_detections),
            'total_epochs': len(stage_indices)
        }

print("\nLow Chin Tone Detection Statistics by Sleep Stage:")
print("-" * 60)
for stage, stats in stage_detection_stats.items():
    print(f"\n{stage_dict[stage]}:")
    print(f"  Mean low tone segments per epoch: {stats['mean_segments']:.2f} ± {stats['std_segments']:.2f}")
    print(f"  Range: [{stats['min_segments']}, {stats['max_segments']}]")
    print(f"  Total epochs analyzed: {stats['total_epochs']}")

#%%
# Final visualization: Detection rate by sleep stage
fig, ax = plt.subplots(figsize=(10, 6))

stages = list(stage_detection_stats.keys())
means = [stage_detection_stats[s]['mean_segments'] for s in stages]
stds = [stage_detection_stats[s]['std_segments'] for s in stages]
stage_names = [stage_dict[s] for s in stages]

bars = ax.bar(stage_names, means, yerr=stds, capsize=5, alpha=0.7, edgecolor='black')
for bar, color in zip(bars, sns.color_palette("husl", len(stages))):
    bar.set_facecolor(color)

ax.set_xlabel('Sleep Stage')
ax.set_ylabel('Mean Number of Low Chin Tone Segments (out of 10)')
ax.set_title('Average Low Chin Tone Detection by Sleep Stage')
ax.grid(True, alpha=0.3)

# Add horizontal line at 5 (50% detection rate)
ax.axhline(5, color='red', linestyle='--', alpha=0.5, label='50% detection rate')
ax.legend()

plt.tight_layout()
plt.show()

print("\nAnalysis complete!")
